# STEP 1 -- CONFIG (all settings in one place)

Edit this cell, re-run it, then re-run from whatever step
you want. Every other cell reads from these variables.

CSV chain (each step loads previous, saves next):
  step2_raw.csv
  step3_clean.csv
  step4_pain.csv
  step5_engagement.csv
  step6_sentiment.csv
  step7_buildable.csv
  step8_filtered.csv
  step9_grouped.csv
  step10_scored.csv
  step11_final.csv

In [ ]:
import os
from datetime import datetime

# ── Run ID (ties all CSVs from one run together) ─────────────
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M")

# ── Reddit scraping ──────────────────────────────────────────
SUBREDDITS = [
    # Business / SMB
    "smallbusiness", "Entrepreneur", "startups", "ecommerce",
    "sweatystartup", "EntrepreneurRideAlong",

    # SaaS / Indie
    "SaaS", "indiehackers", "microsaas", "SideProject",

    # Tech / Dev
    "webdev", "selfhosted", "devops", "sysadmin",
    "reactjs", "django", "node", "nextjs",

    # Automation / Ops
    "automation", "nocode", "lowcode",

    # Freelance / Agency
    "freelance", "webdesign", "digital_marketing",

    # Side income
    "sidehustle", "passive_income",

    # Niche verticals (where people complain about tools)
    "sales", "CRM", "customerservice",
    "projectmanagement", "productivity",
    "bookkeeping", "invoicing",
    "emailmarketing", "analytics",
    "dataengineering", "datascience",
]

LOOKBACK_DAYS       = 90         # how far back (Reddit caps at ~1000 posts/sub)
PAGES_PER_SUB       = 10         # pages x 100 = max posts per sub
REDDIT_SLEEP        = 2.5       # seconds between requests
FETCH_COMMENTS      = True      # pull top comments per post (slower but richer)
TOP_COMMENTS_N      = 5         # how many top comments to grab per post

# ── Pain keywords (Step 4) ───────────────────────────────────
GATE_KEYWORDS = [
    "how to", "how do i", "is there a", "is there any",
    "issue", "problem", "frustrated", "struggling", "annoying",
    "tool for", "looking for", "alternative to", "switch from",
    "anyone know", "any recommendations", "recommend", "suggestion",
    "manually", "waste time", "need help", "better way",
    "too expensive", "wish there was", "hate using", "sick of",
    "pain", "headache", "nightmare", "broken",
    "need a tool", "need software", "need an app",
    "takes too long", "hours every", "tedious",
]

PAIN_SIGNALS = {
    "willing to pay": 35, "take my money": 35,
    "looking for tool": 28, "looking for software": 28,
    "looking for app": 28, "need a tool": 25,
    "alternative to": 25, "switch from": 25,
    "wish there was": 25,
    "hate": 14, "frustrated": 14, "sick of": 14, "fed up": 14,
    "annoying": 10, "nightmare": 16, "headache": 12,
    "waste time": 16, "too much time": 16, "hours every": 16,
    "tedious": 12, "painful": 12, "broken": 10,
    "manually": 12, "spreadsheet": 16, "excel": 14,
    "copy paste": 14, "by hand": 12,
    "recommendations": 14, "anyone know": 10, "does anyone": 8,
    "better way": 12, "any suggestions": 8, "need help": 6,
}

MIN_PAIN_SCORE = 6  # keep this low; later steps filter harder

# ── Buildability keywords (Step 7) ───────────────────────────
SOFTWARE_SIGNALS = {
    "api": 15, "automate": 15, "automation": 15,
    "dashboard": 12, "integrate": 12, "integration": 12,
    "export": 10, "sync": 12, "webhook": 15,
    "workflow": 12, "pipeline": 10,
    "spreadsheet": 10, "excel": 10, "csv": 10,
    "notification": 8, "alert": 8, "monitor": 10,
    "template": 8, "generate": 8, "schedule": 10,
    "track": 8, "report": 10,
    "manual": 8, "manually": 8, "repetitive": 10,
    "copy paste": 12, "data entry": 12,
    "time consuming": 8, "hours every": 10,
    "tool": 5, "software": 5, "app": 3, "platform": 3,
    "saas": 5, "chrome extension": 8, "plugin": 8,
    "bot": 8, "scrape": 8,
}

ADVICE_PENALTIES = {
    "advice": -8, "guidance": -8, "tips": -6,
    "how do i start": -10, "where do i start": -10,
    "is it worth": -8, "should i": -6,
    "first sale": -10, "first client": -10,
    "cold email": -5, "cold call": -5, "outreach": -5,
    "networking": -8, "word of mouth": -8,
    "social media marketing": -6, "content marketing": -6,
}

MIN_BUILDABILITY = 0  # Step 7 threshold

# ── Topic exclusions (Step 8) ────────────────────────────────
EXCLUDE_TOPICS = [
    "restaurant", "food truck", "cafe", "bakery", "catering",
    "bar ", "bartend", "food delivery", "meal prep",
    "gaming", "game dev", "unity", "unreal", "steam",
    "bank", "banking", "loan", "mortgage", "credit score", "credit card",
    "real estate", "rental property", "landlord", "tenant", "airbnb",
    "crypto", "nft", "blockchain", "web3", "defi",
    "dropship", "print on demand", "amazon fba", "tiktok shop",
    "vpn", "hosting provider",
    "insurance", "healthcare", "medical", "hipaa",
    "legal", "law firm", "attorney", "lawyer",
    "construction", "plumbing", "hvac", "roofing", "landscaping",
    "cleaning business", "pressure wash", "lawn care",
    "trucking", "logistics", "shipping",
    "music production", "beat", "podcast",
    "wedding", "photography business", "videography",
    "fitness", "gym", "personal trainer", "yoga",
    "pet", "dog walking", "grooming", "tutoring", "coaching business",
    "google merchant", "google ads", "google analytics",
    "shopify app", "shopify plugin", "shopify theme",
    "wordpress plugin", "wordpress theme", "woocommerce",
    "salesforce", "hubspot", "quickbooks", "xero",
    "zapier", "make.com", "notion template", "airtable",
    "hire", "hiring", "finding employees", "staffing", "recruiting",
    "fired", "laid off", "lost my job", "need money", "need income",
    "can't afford", "debt", "broke", "financial situation",
    "visa", "immigration", "work permit",
    "tax", "taxes", "accounting", "bookkeep",
    "i quit", "i'm done", "giving up", "burned out", "burnout",
    "impostor syndrome", "mental health", "anxiety", "depressed",
    "rant", "vent",
    "here is how i", "here's how i", "how i made",
    "how i built", "i made $", "i earned", "i made over",
    "$k/month", "$k per month", "$k/mo",
    "case study", "success story",
]

EXCLUDE_TITLE_PATTERNS = [
    r"^ama\b", r"\bshare your\b", r"\bmy journey\b",
    r"\bfree promo\b", r"\bi will make\b",
    r"\blooking for co-?founder\b", r"\bhiring\b",
    r"\bjob posting\b", r"\bintern(ship)?\b",
    r"\bweekly thread\b", r"\bmonthly thread\b",
    r"\bwho wants to\b", r"\bfeedback friday\b",
    r"\blaunch(ed|ing)?\s+(my|our|a|the)\b",
    r"\bjust (launched|shipped|released)\b",
]

NOT_SOFTWARE_PATTERNS = [
    r"\bhow (do|can|should) (i|we|you) (get|find|attract|reach) (clients|customers|users|leads|sales)\b",
    r"\bhow (do|can|should) (i|we|you) (start|launch|grow|scale|market|sell|promote|advertise)\b",
    r"\bhow (do|can) (i|we) (make money|earn|monetize|charge|price)\b",
    r"\b(advice|guidance|tips) (on|for|about) (starting|running|growing|marketing|selling)\b",
    r"\bwhat (business|side hustle|venture) (should|can|could) i\b",
    r"\b(first sale|first client|first customer|first user|first paying)\b",
    r"\bhow to (get|find|close) (deal|sale|contract|client)\b",
    r"\b(finally|just) (made|got|reached|hit) (my|our) first\b",
    r"\bwhat (i|we) learned\b",
    r"\bshould i (quit|leave|stay|switch|pivot)\b",
    r"\b(career|job) (advice|change|switch|path)\b",
    r"\bhow (much|do you|should i) charge\b",
    r"\bno (sales|revenue|customers|traction|users)\b",
    r"\b(marketing|sales) (strategy|channel|tactic|approach)\b",
]

# ── Grouping (Step 9) ────────────────────────────────────────
GROUP_SIMILARITY    = 0.25   # cosine threshold (lower = bigger groups)
MERGE_THRESHOLD     = 0.50   # second-pass merge for near-duplicates

# ── File naming ──────────────────────────────────────────────
def csv_name(step_num, label):
    return f"step{step_num}_{label}_{RUN_ID}.csv"

print(f"Config loaded. Run ID: {RUN_ID}")
print(f"  {len(SUBREDDITS)} subreddits, {LOOKBACK_DAYS}d lookback")
print(f"  Comments: {'ON' if FETCH_COMMENTS else 'OFF'}")
print(f"  CSV prefix: step<N>_<label>_{RUN_ID}.csv")

# STEP 2 -- INSTALL + VERIFY ENVIRONMENT

Installs all packages and verifies imports work.
No API calls, no tokens, no network beyond pip.

Fields used: none (setup only)
Fields added: none
CSV: none

In [ ]:
import json, time, re, requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from tqdm.auto import tqdm
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("[OK] All imports successful")
print(f"  pandas {pd.__version__}")
print(f"  numpy {np.__version__}")

# STEP 3 -- SCRAPE REDDIT

Fetches posts from all SUBREDDITS within LOOKBACK_DAYS.
For each post, also fetches top comments (if FETCH_COMMENTS).

Reddit JSON API, no OAuth needed. Paginates with ?after=.
Handles 429 rate limits. Deduplicates by post_id.

Fields extracted per post:
  post_id, subreddit, title, body, url,
  upvotes, upvote_ratio, num_comments,
  created_utc, post_age_hours,
  flair,
  top_comments_text     (joined text of top N comments)
  top_comments_scores   (comma-sep scores of top N comments)
  top_comment_count     (how many comments we actually got)

Saves: step3_raw_{RUN_ID}.csv

In [ ]:
STEP3_CSV = csv_name(3, "raw")

headers = {"User-Agent": "SaaSResearch/4.0 (academic; contact@example.com)"}
cutoff = time.time() - (LOOKBACK_DAYS * 86400)
cutoff_str = datetime.utcfromtimestamp(cutoff).strftime("%Y-%m-%d")

print(f"Scraping posts after {cutoff_str} ({LOOKBACK_DAYS}d)")
print(f"Subs: {len(SUBREDDITS)}, pages/sub: {PAGES_PER_SUB}")
print(f"Comments: {'ON (top {})'.format(TOP_COMMENTS_N) if FETCH_COMMENTS else 'OFF'}\n")

seen = set()
rows = []

def fetch_top_comments(permalink, n=TOP_COMMENTS_N):
    """
    Fetch top N comments for a post by score.

    Args:
        permalink: Reddit post permalink (e.g. /r/SaaS/comments/abc123/...)
        n:         Number of top comments to return

    Returns:
        list of dicts: [{"text": "...", "score": 42}, ...]
    """
    url = f"https://www.reddit.com{permalink}.json?limit={n + 5}&sort=top&raw_json=1"
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code != 200:
            return []

        data = r.json()
        if len(data) < 2:
            return []

        comments_data = data[1].get("data", {}).get("children", [])
        results = []
        for c in comments_data:
            if c.get("kind") != "t1":
                continue
            cd = c["data"]
            body = (cd.get("body") or "").strip()
            if not body or body == "[deleted]" or body == "[removed]":
                continue
            results.append({
                "text": body[:500],
                "score": cd.get("score", 0),
            })

        # Sort by score descending, take top N
        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:n]

    except Exception:
        return []


for sub in tqdm(SUBREDDITS, desc="Subreddits"):
    after_token = None
    sub_count = 0

    for page in range(PAGES_PER_SUB):
        params = {"limit": 100, "raw_json": 1}
        if after_token:
            params["after"] = after_token

        try:
            r = requests.get(
                f"https://www.reddit.com/r/{sub}/new.json",
                headers=headers, params=params, timeout=15,
            )

            if r.status_code == 429:
                wait = int(r.headers.get("Retry-After", 15))
                tqdm.write(f"  r/{sub}: 429, sleeping {wait}s")
                time.sleep(wait)
                continue
            if r.status_code != 200:
                tqdm.write(f"  r/{sub}: HTTP {r.status_code}")
                break

            data = r.json().get("data", {})
            children = data.get("children", [])
            if not children:
                break

            for child in children:
                d = child["data"]
                if d["created_utc"] < cutoff:
                    continue
                if d["id"] in seen:
                    continue
                seen.add(d["id"])

                title = (d.get("title") or "").strip()
                if not title:
                    continue

                body = (d.get("selftext") or "").strip()

                # Fetch comments if enabled
                comments_text = ""
                comments_scores = ""
                comment_count = 0

                if FETCH_COMMENTS and d.get("num_comments", 0) > 0:
                    top_comments = fetch_top_comments(d["permalink"])
                    if top_comments:
                        comments_text = " ||| ".join(c["text"] for c in top_comments)
                        comments_scores = ", ".join(str(c["score"]) for c in top_comments)
                        comment_count = len(top_comments)
                    time.sleep(0.5)  # extra pause for comment fetches

                rows.append({
                    "post_id":              d["id"],
                    "subreddit":            sub,
                    "title":                title,
                    "body":                 body[:5000],
                    "url":                  f"https://reddit.com{d['permalink']}",
                    "upvotes":              d.get("ups", 0),
                    "upvote_ratio":         d.get("upvote_ratio", 0.5),
                    "num_comments":         d.get("num_comments", 0),
                    "created_utc":          d["created_utc"],
                    "post_age_hours":       round((time.time() - d["created_utc"]) / 3600, 1),
                    "flair":                d.get("link_flair_text") or "",
                    "top_comments_text":    comments_text[:5000],
                    "top_comments_scores":  comments_scores,
                    "top_comment_count":    comment_count,
                })
                sub_count += 1

            after_token = data.get("after")
            if not after_token:
                break

        except requests.exceptions.Timeout:
            tqdm.write(f"  r/{sub} p{page}: timeout")
            break
        except Exception as e:
            tqdm.write(f"  r/{sub} p{page}: {type(e).__name__}")
            break

        time.sleep(REDDIT_SLEEP)

    tqdm.write(f"  r/{sub}: {sub_count}")

df_raw = pd.DataFrame(rows)
df_raw.to_csv(STEP3_CSV, index=False)

print(f"\n{'='*55}")
print(f" SCRAPED: {len(df_raw):,} posts --> {STEP3_CSV}")
print(f"{'='*55}")
if not df_raw.empty:
    for sub, n in df_raw["subreddit"].value_counts().items():
        print(f"  r/{sub:25s} {n:4d}")
    print(f"\n  Age: {df_raw['post_age_hours'].min():.0f}h -- {df_raw['post_age_hours'].max():.0f}h")
    print(f"  Median upvotes:  {df_raw['upvotes'].median():.0f}")
    print(f"  Median comments: {df_raw['num_comments'].median():.0f}")
    if FETCH_COMMENTS:
        has_comments = (df_raw["top_comment_count"] > 0).sum()
        print(f"  Posts with comments fetched: {has_comments}")

# STEP 4 -- CLEAN + BUILD FULL TEXT

Loads raw CSV, cleans text, builds a combined searchable
column that includes title + body + top comments.

This step does NOT filter anything. It just prepares the
text so all subsequent steps can work with one column.

Fields added:
  full_text       title + body + top comments, cleaned
  text_length     character count of full_text
  has_body        whether the post has selftext
  has_comments    whether we have comment text

Loads:  step3_raw_{RUN_ID}.csv
Saves:  step4_clean_{RUN_ID}.csv

In [ ]:
import re

STEP3_CSV = csv_name(3, "raw")
STEP4_CSV = csv_name(4, "clean")

df = pd.read_csv(STEP3_CSV)
print(f"Loaded {STEP3_CSV}: {len(df):,} rows")

# ── Clean text ───────────────────────────────────────────────
def clean_text(text):
    """Remove markdown artifacts, URLs, excessive whitespace."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'https?://\S+', '', text)           # URLs
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)  # [text](url) -> text
    text = re.sub(r'[#*_~`>]', '', text)                # markdown chars
    text = re.sub(r'\s+', ' ', text).strip()             # collapse whitespace
    return text

df["title"] = df["title"].apply(clean_text)
df["body"] = df["body"].fillna("").apply(clean_text)
df["top_comments_text"] = df["top_comments_text"].fillna("").apply(clean_text)

# ── Build combined text ──────────────────────────────────────
# title + body + comments = one searchable column
df["full_text"] = (
    df["title"] + " " + df["body"] + " " + df["top_comments_text"]
).str.strip()

df["text_length"] = df["full_text"].str.len()
df["has_body"] = (df["body"].str.len() > 10).astype(int)
df["has_comments"] = (df["top_comments_text"].str.len() > 10).astype(int)

df.to_csv(STEP4_CSV, index=False)

print(f"\n{'='*55}")
print(f" CLEANED: {len(df):,} posts --> {STEP4_CSV}")
print(f"{'='*55}")
print(f"  Avg text length: {df['text_length'].mean():.0f} chars")
print(f"  Posts with body:     {df['has_body'].sum():,}")
print(f"  Posts with comments: {df['has_comments'].sum():,}")

# STEP 5 -- PAIN KEYWORD SCORING

In [ ]:
# Two stages:
#   A) Gate: post must contain >= 1 GATE_KEYWORD to survive.
#      This is a broad filter to remove totally off-topic posts.
#   B) Score: each surviving post gets a pain_score based on
#      which PAIN_SIGNALS it matches. Scores are additive.
#
# We search the full_text (title + body + comments) so pain
# signals in comments also count. E.g. someone replies
# "I waste so much time on this" boosts the post.
#
# Fields added:
#   pain_score        sum of matched signal weights
#   matched_signals   comma-sep list of which signals hit
#
# Fields used: full_text, GATE_KEYWORDS, PAIN_SIGNALS
#
# Loads:  step4_clean_{RUN_ID}.csv
# Saves:  step5_pain_{RUN_ID}.csv

In [ ]:
STEP4_CSV = csv_name(4, "clean")
STEP5_CSV = csv_name(5, "pain")

df = pd.read_csv(STEP4_CSV)
print(f"Loaded {STEP4_CSV}: {len(df):,} rows")

ft = df["full_text"].fillna("").str.lower()

# ── Stage A: Gate filter ─────────────────────────────────────
gate_mask = pd.Series(False, index=df.index)
for kw in GATE_KEYWORDS:
    gate_mask |= ft.str.contains(kw, regex=False)

before = len(df)
df = df.loc[gate_mask].copy()
print(f"Gate filter: {before:,} --> {len(df):,}  (-{before - len(df):,})")

# ── Stage B: Pain scoring ────────────────────────────────────
ft = df["full_text"].fillna("").str.lower()

scores = pd.Series(0.0, index=df.index)
signals = pd.Series("", index=df.index)

for signal, weight in PAIN_SIGNALS.items():
    mask = ft.str.contains(signal, regex=False)
    scores += mask.astype(float) * weight
    signals = signals.where(~mask, signals + signal + ", ")

df["pain_score"] = scores.round(1)
df["matched_signals"] = signals.str.rstrip(", ")

# Cut below threshold
before = len(df)
df = df.loc[df["pain_score"] >= MIN_PAIN_SCORE].copy()
df = df.sort_values("pain_score", ascending=False).reset_index(drop=True)
print(f"Pain floor ({MIN_PAIN_SCORE}): {before:,} --> {len(df):,}  (-{before - len(df):,})")

df.to_csv(STEP5_CSV, index=False)

print(f"\n{'='*55}")
print(f" PAIN SCORED: {len(df):,} posts --> {STEP5_CSV}")
print(f"{'='*55}")
print(f"  Score range: {df['pain_score'].min():.0f} -- {df['pain_score'].max():.0f}")
print(f"  Top 5:")
for _, r in df.head(5).iterrows():
    print(f"  [{r['pain_score']:5.1f}] r/{r['subreddit']:18s} {r['title'][:60]}")
    print(f"          signals: {r['matched_signals'][:70]}")

# STEP 6 -- ENGAGEMENT SCORING

Measures how much the community validated this problem.

Signals used:
  - upvotes:       direct agreement
  - upvote_ratio:  high ratio = genuine (not controversial)
  - num_comments:  discussion = people care
  - top comment scores: high-scored replies = strong resonance

Formula uses log scaling so a 500-upvote post doesn't
completely dominate a 20-upvote post that's more relevant.

A post with 5 upvotes but 40 comments is likely a heated
discussion about a real problem. The comment weight
reflects this.

Fields added:
  engagement_score    composite engagement metric
  comment_quality     avg score of top comments (if available)
  controversy         1 - upvote_ratio (higher = more divisive)

Fields used: upvotes, upvote_ratio, num_comments,
             top_comments_scores

Loads:  step5_pain_{RUN_ID}.csv
Saves:  step6_engagement_{RUN_ID}.csv

In [ ]:
STEP5_CSV = csv_name(5, "pain")
STEP6_CSV = csv_name(6, "engagement")

df = pd.read_csv(STEP5_CSV)
print(f"Loaded {STEP5_CSV}: {len(df):,} rows")

# ── Comment quality score ─────────────────────────────────────
def avg_comment_score(scores_str):
    """Parse comma-separated comment scores, return mean."""
    if not isinstance(scores_str, str) or not scores_str.strip():
        return 0.0
    try:
        vals = [float(x.strip()) for x in scores_str.split(",") if x.strip()]
        return np.mean(vals) if vals else 0.0
    except ValueError:
        return 0.0

df["comment_quality"] = df["top_comments_scores"].apply(avg_comment_score)

# ── Controversy (low ratio = divisive topic, still interesting)
df["controversy"] = (1 - df["upvote_ratio"].fillna(0.5)).round(3)

# ── Composite engagement score ────────────────────────────────
# log-scaled so outliers don't dominate
df["engagement_score"] = (
    np.log1p(df["upvotes"]) * 2.5             # upvotes
    + np.log1p(df["num_comments"]) * 3.5      # comments (weighted more -- discussion = real pain)
    + np.log1p(df["comment_quality"]) * 2.0   # quality of responses
    + df["upvote_ratio"].fillna(0.5) * 3.0    # consensus bonus
).round(1)

df.to_csv(STEP6_CSV, index=False)

print(f"\n{'='*55}")
print(f" ENGAGEMENT SCORED: {len(df):,} posts --> {STEP6_CSV}")
print(f"{'='*55}")
print(f"  Engagement range: {df['engagement_score'].min():.1f} -- {df['engagement_score'].max():.1f}")
print(f"  Avg comment quality: {df['comment_quality'].mean():.1f}")
print(f"  Top 5 by engagement:")
for _, r in df.nlargest(5, "engagement_score").iterrows():
    print(f"  [{r['engagement_score']:5.1f}] ^{r['upvotes']:4d} {r['num_comments']:3d}c  r/{r['subreddit']:15s} {r['title'][:55]}")

# STEP 7 -- SENTIMENT + BUYING INTENT (NLP, no API)

TextBlob sentiment analysis on the full text (title + body
+ comments). Negative sentiment = genuine frustration.

Buying intent is a separate score: does the person sound
like they're actively searching for a solution to pay for?
"Looking for a tool" vs "this is annoying" are both pain
but only one leads to a purchase.

Fields added:
  sentiment          TextBlob polarity (-1 to +1)
  subjectivity       TextBlob subjectivity (0 to 1)
  sentiment_label    negative / neutral / positive
  buying_intent      0-100 score based on purchase-signal phrases

Fields used: full_text, title

Loads:  step6_engagement_{RUN_ID}.csv
Saves:  step7_sentiment_{RUN_ID}.csv

In [ ]:
STEP6_CSV = csv_name(6, "engagement")
STEP7_CSV = csv_name(7, "sentiment")

df = pd.read_csv(STEP6_CSV)
print(f"Loaded {STEP6_CSV}: {len(df):,} rows")

# ── Sentiment ─────────────────────────────────────────────────
print("Running TextBlob sentiment (this takes a minute)...")

sentiments = df["full_text"].fillna("").apply(
    lambda t: TextBlob(t).sentiment
)
df["sentiment"] = sentiments.apply(lambda s: round(s.polarity, 3))
df["subjectivity"] = sentiments.apply(lambda s: round(s.subjectivity, 3))

df["sentiment_label"] = pd.cut(
    df["sentiment"],
    bins=[-1.01, -0.1, 0.1, 1.01],
    labels=["negative", "neutral", "positive"],
)

# ── Buying intent ─────────────────────────────────────────────
# Weighted phrases that signal readiness to pay
BUYING_PHRASES = {
    "willing to pay":       30,
    "take my money":        30,
    "looking for tool":     25,
    "looking for software": 25,
    "looking for app":      25,
    "need a tool":          22,
    "alternative to":       20,
    "switch from":          20,
    "wish there was":       20,
    "anyone know a":        15,
    "recommendations":      12,
    "any recommendations":  15,
    "does anyone know":     12,
    "is there a":           10,
    "is there any":         10,
    "better than":          12,
    "replace":              10,
    "budget for":           18,
    "what do you use":      12,
    "what tool":            15,
    "what software":        15,
    "what app":             15,
}

ft = df["full_text"].fillna("").str.lower()
intent = pd.Series(0.0, index=df.index)
for phrase, weight in BUYING_PHRASES.items():
    intent += ft.str.contains(phrase, regex=False).astype(float) * weight

df["buying_intent"] = intent.clip(upper=100).round(1)

df.to_csv(STEP7_CSV, index=False)

print(f"\n{'='*55}")
print(f" SENTIMENT + INTENT: {len(df):,} posts --> {STEP7_CSV}")
print(f"{'='*55}")
print(f"  Sentiment: {df['sentiment_label'].value_counts().to_dict()}")
print(f"  Avg subjectivity: {df['subjectivity'].mean():.2f}")
print(f"  Buying intent > 0: {(df['buying_intent'] > 0).sum():,}")
print(f"\n  Top 5 buying intent:")
for _, r in df.nlargest(5, "buying_intent").iterrows():
    print(f"  [{r['buying_intent']:5.1f}] r/{r['subreddit']:15s} {r['title'][:60]}")

# STEP 8 -- BUILDABILITY SCORING

Scores how likely this problem is solvable with software.

Posts mentioning APIs, automation, dashboards, workflows
get boosted. Posts seeking general advice, networking,
marketing tips get penalized.

Removes posts with buildability below MIN_BUILDABILITY.

Fields added:
  buildability    software-signal score (can be negative)

Fields used: full_text, SOFTWARE_SIGNALS, ADVICE_PENALTIES

Loads:  step7_sentiment_{RUN_ID}.csv
Saves:  step8_buildable_{RUN_ID}.csv

In [ ]:
STEP7_CSV = csv_name(7, "sentiment")
STEP8_CSV = csv_name(8, "buildable")

df = pd.read_csv(STEP7_CSV)
print(f"Loaded {STEP7_CSV}: {len(df):,} rows")

ft = df["full_text"].fillna("").str.lower()

build = pd.Series(0.0, index=df.index)
for term, w in {**SOFTWARE_SIGNALS, **ADVICE_PENALTIES}.items():
    build += ft.str.contains(term, regex=False).astype(float) * w

df["buildability"] = build.round(1)

before = len(df)
df = df.loc[df["buildability"] >= MIN_BUILDABILITY].copy()
print(f"Buildability filter (>= {MIN_BUILDABILITY}): {before:,} --> {len(df):,}  (-{before - len(df):,})")

df.to_csv(STEP8_CSV, index=False)

print(f"\n{'='*55}")
print(f" BUILDABLE: {len(df):,} posts --> {STEP8_CSV}")
print(f"{'='*55}")
print(f"  Buildability range: {df['buildability'].min():.0f} -- {df['buildability'].max():.0f}")
print(f"  Top 5:")
for _, r in df.nlargest(5, "buildability").iterrows():
    print(f"  [{r['buildability']:5.1f}] r/{r['subreddit']:15s} {r['title'][:60]}")

# STEP 9 -- TOPIC + PATTERN EXCLUSIONS

Removes posts matching excluded topics (industries you
don't want), title patterns (AMAs, promos, launches),
and not-software patterns (business advice seeking).

This is a hard filter. Anything matching gets dropped.
Edit EXCLUDE_TOPICS, EXCLUDE_TITLE_PATTERNS, and
NOT_SOFTWARE_PATTERNS in Step 1 to control this.

Fields added: none (filter only)
Fields used:  full_text, title

Loads:  step8_buildable_{RUN_ID}.csv
Saves:  step9_filtered_{RUN_ID}.csv

In [ ]:
STEP8_CSV = csv_name(8, "buildable")
STEP9_CSV = csv_name(9, "filtered")

df = pd.read_csv(STEP8_CSV)
print(f"Loaded {STEP8_CSV}: {len(df):,} rows")

ft = df["full_text"].fillna("").str.lower()

# ── A: Topic exclusions ──────────────────────────────────────
topic_mask = pd.Series(False, index=df.index)
for topic in EXCLUDE_TOPICS:
    topic_mask |= ft.str.contains(topic.lower(), regex=False)

before = len(df)
df = df.loc[~topic_mask].copy()
print(f"Topics:         {before:,} --> {len(df):,}  (-{before - len(df):,})")

# ── B: Title patterns ────────────────────────────────────────
before = len(df)
pat_mask = pd.Series(False, index=df.index)
for pat in EXCLUDE_TITLE_PATTERNS:
    pat_mask |= df["title"].fillna("").str.lower().str.contains(pat, regex=True, na=False)
df = df.loc[~pat_mask].copy()
print(f"Title patterns: {before:,} --> {len(df):,}  (-{before - len(df):,})")

# ── C: Not-software patterns ─────────────────────────────────
before = len(df)
ft = df["full_text"].fillna("").str.lower()
sw_mask = pd.Series(False, index=df.index)
for pat in NOT_SOFTWARE_PATTERNS:
    sw_mask |= ft.str.contains(pat, regex=True, na=False)
df = df.loc[~sw_mask].copy()
print(f"Not-software:   {before:,} --> {len(df):,}  (-{before - len(df):,})")

df = df.reset_index(drop=True)
df.to_csv(STEP9_CSV, index=False)

print(f"\n{'='*55}")
print(f" FILTERED: {len(df):,} posts --> {STEP9_CSV}")
print(f"{'='*55}")
if not df.empty:
    print(f"  By subreddit:")
    for sub, n in df["subreddit"].value_counts().head(8).items():
        print(f"    r/{sub:22s} {n:4d}")

# STEP 10 -- GROUP SIMILAR POSTS

Clusters posts about the same problem using TF-IDF cosine
similarity on titles. Posts with similar titles get merged
into one group.

Why group AFTER filtering (not before):
  - Fewer posts = faster TF-IDF + less noise in clusters
  - Excluded posts don't pollute groups
  - Groups are cleaner and more meaningful

How grouping works:
  1. TF-IDF vectorize all titles (bag of 1- and 2-grams)
  2. Compute pairwise cosine similarity matrix
  3. Greedy clustering seeded by highest-scored posts:
     - Pick the highest-scored ungrouped post
     - Pull in all ungrouped posts with sim >= GROUP_SIMILARITY
     - Mark all as assigned, repeat
  4. Second pass: merge groups whose rep_titles are
     >= MERGE_THRESHOLD similar

Each group row contains:
  - Aggregated stats (post count, total engagement, etc.)
  - The highest-scored post's title as representative
  - All member post_ids for traceability
  - A group_name column for easy filtering
  - The combined text of all member titles + best body
    (for downstream analysis)

A group with 5 posts about the same issue is stronger
evidence than a single post with 50 upvotes. The
validation_score reflects this (added in Step 11).

Fields per group:
  group_id, post_count, post_ids,
  rep_title, combined_text,
  top_url, total_upvotes, total_comments,
  avg_pain_score, max_pain_score,
  avg_engagement, avg_buying_intent, avg_buildability,
  avg_sentiment, top_subreddits, top_signals,
  sample_titles

Loads:  step9_filtered_{RUN_ID}.csv
Saves:  step10_grouped_{RUN_ID}.csv

In [ ]:
STEP9_CSV = csv_name(9, "filtered")
STEP10_CSV = csv_name(10, "grouped")

df = pd.read_csv(STEP9_CSV)
print(f"Loaded {STEP9_CSV}: {len(df):,} posts")

assert len(df) > 0, "No posts to group."

# ── TF-IDF + greedy clustering ────────────────────────────────
titles = df["title"].fillna("").tolist()
tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
matrix = tfidf.fit_transform(titles)
sim = cosine_similarity(matrix)

assigned = set()
clusters = []
order = df["pain_score"].values.argsort()[::-1]

for idx in order:
    if idx in assigned:
        continue
    neighbors = [j for j in range(len(df)) if j not in assigned and sim[idx, j] >= GROUP_SIMILARITY]
    assigned.update(neighbors)
    clusters.append(neighbors)

print(f"Initial clusters: {len(clusters)}")

# ── Build group rows ──────────────────────────────────────────
group_rows = []
for gid, member_idxs in enumerate(clusters):
    members = df.iloc[member_idxs]
    best = members.loc[members["pain_score"].idxmax()]

    # Signals aggregation
    all_sigs = ", ".join(members["matched_signals"].dropna()).split(", ")
    all_sigs = [s.strip() for s in all_sigs if s.strip()]
    sig_counts = pd.Series(all_sigs).value_counts() if all_sigs else pd.Series(dtype=int)

    # Subreddit distribution
    sub_dist = members["subreddit"].value_counts()
    top_subs = ", ".join(f"{s}({n})" for s, n in sub_dist.head(4).items())

    # Combined text
    combined = "\n".join(f"- {t}" for t in members["title"].tolist())
    if isinstance(best["body"], str) and len(best["body"]) > 20:
        combined += f"\n\nBest post body:\n{best['body'][:2000]}"

    # Sample titles for inspection
    samples = " | ".join(members["title"].head(3).tolist())

    group_rows.append({
        "group_id":           gid,
        "post_count":         len(members),
        "post_ids":           ", ".join(members["post_id"].astype(str).tolist()),
        "rep_title":          best["title"],
        "combined_text":      combined[:5000],
        "sample_titles":      samples[:300],
        "top_url":            best["url"],
        "total_upvotes":      int(members["upvotes"].sum()),
        "total_comments":     int(members["num_comments"].sum()),
        "avg_pain_score":     round(members["pain_score"].mean(), 1),
        "max_pain_score":     round(members["pain_score"].max(), 1),
        "avg_engagement":     round(members["engagement_score"].mean(), 1),
        "avg_buying_intent":  round(members["buying_intent"].mean(), 1),
        "avg_buildability":   round(members["buildability"].mean(), 1),
        "avg_sentiment":      round(members["sentiment"].mean(), 3),
        "top_subreddits":     top_subs,
        "top_signals":        ", ".join(sig_counts.head(5).index.tolist()),
    })

df_g = pd.DataFrame(group_rows)

# ── Second pass: merge near-duplicate groups ──────────────────
if len(df_g) > 1:
    tfidf2 = TfidfVectorizer(max_features=3000, stop_words="english", ngram_range=(1, 2))
    mat2 = tfidf2.fit_transform(df_g["rep_title"].fillna(""))
    sim2 = cosine_similarity(mat2)

    merged = {}
    idxs = df_g.index.tolist()
    for i_pos in range(len(idxs)):
        i = idxs[i_pos]
        if i in merged:
            continue
        for j_pos in range(i_pos + 1, len(idxs)):
            j = idxs[j_pos]
            if j in merged:
                continue
            if sim2[i_pos, j_pos] >= MERGE_THRESHOLD:
                merged[j] = i

    for child, parent in merged.items():
        if parent in df_g.index and child in df_g.index:
            for col in ["post_count", "total_upvotes", "total_comments"]:
                df_g.at[parent, col] += df_g.at[child, col]
            df_g.at[parent, "post_ids"] = str(df_g.at[parent, "post_ids"]) + ", " + str(df_g.at[child, "post_ids"])
            for col in ["avg_pain_score", "avg_engagement", "avg_buying_intent", "avg_buildability"]:
                df_g.at[parent, col] = max(df_g.at[parent, col], df_g.at[child, col])

    before_merge = len(df_g)
    df_g = df_g.drop(index=[c for c in merged.keys() if c in df_g.index])
    print(f"Merge pass: {before_merge} --> {len(df_g)} (-{before_merge - len(df_g)})")

df_g = df_g.reset_index(drop=True)
df_g.to_csv(STEP10_CSV, index=False)

print(f"\n{'='*55}")
print(f" GROUPED: {len(df_g):,} groups --> {STEP10_CSV}")
print(f"{'='*55}")
multi = df_g[df_g["post_count"] > 1]
print(f"  Multi-post groups: {len(multi)} (validated)")
print(f"  Single-post groups: {len(df_g) - len(multi)}")
print(f"\n  Top 10:")
for _, g in df_g.nlargest(10, "avg_pain_score").iterrows():
    print(f"  [{g['avg_pain_score']:5.1f}] ({g['post_count']}x) {g['rep_title'][:70]}")

# STEP 11 -- FINAL COMPOSITE SCORING + EXPORT

Combines all dimensions into one final_score per group.
No LLM. Pure math on the signals we've built up.

Score formula:
  pain_component       = avg_pain_score (keyword signals)
  engagement_component = log(total_engagement) (popularity)
  intent_component     = avg_buying_intent (purchase signals)
  build_component      = avg_buildability (software solvable)
  validation_component = post_count (multiple people = real)
  sentiment_component  = negative sentiment boost

These are normalized to 0-1 range then weighted:
  Pain:        25%  (is it a real problem?)
  Engagement:  20%  (do people care?)
  Intent:      20%  (will they pay?)
  Buildability:15%  (can I build it?)
  Validation:  15%  (multiple sources?)
  Sentiment:    5%  (genuinely negative?)

Final score is 0-100.

Fields added:
  final_score          0-100 composite
  pain_norm, eng_norm, intent_norm, build_norm,
  valid_norm, sent_norm    (individual 0-1 components)

Loads:  step10_grouped_{RUN_ID}.csv
Saves:  step11_final_{RUN_ID}.csv

In [ ]:
STEP10_CSV = csv_name(10, "grouped")
STEP11_CSV = csv_name(11, "final")

df = pd.read_csv(STEP10_CSV)
print(f"Loaded {STEP10_CSV}: {len(df):,} groups")


def normalize_col(series, cap_percentile=95):
    """
    Normalize a series to 0-1 range, capping outliers at
    the given percentile so one viral post doesn't dominate.
    """
    cap = series.quantile(cap_percentile / 100)
    capped = series.clip(upper=cap)
    mn, mx = capped.min(), capped.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return ((capped - mn) / (mx - mn)).round(4)


df["total_engagement"] = df["total_upvotes"] + df["total_comments"]

# ── Normalize each dimension to 0-1 ──────────────────────────
df["pain_norm"]    = normalize_col(df["avg_pain_score"])
df["eng_norm"]     = normalize_col(np.log1p(df["total_engagement"]))
df["intent_norm"]  = normalize_col(df["avg_buying_intent"])
df["build_norm"]   = normalize_col(df["avg_buildability"])
df["valid_norm"]   = normalize_col(df["post_count"].clip(upper=20))
df["sent_norm"]    = normalize_col(df["avg_sentiment"].clip(upper=0).abs())

# ── Weighted composite ────────────────────────────────────────
df["final_score"] = (
    df["pain_norm"]   * 25
    + df["eng_norm"]  * 20
    + df["intent_norm"] * 20
    + df["build_norm"] * 15
    + df["valid_norm"] * 15
    + df["sent_norm"]  * 5
).round(1)

df = df.sort_values("final_score", ascending=False).reset_index(drop=True)
df.to_csv(STEP11_CSV, index=False)

# ── Display ───────────────────────────────────────────────────
print(f"\n{'='*65}")
print(f" FINAL: {len(df):,} groups scored --> {STEP11_CSV}")
print(f"{'='*65}")
print(f"  Score range: {df['final_score'].min():.1f} -- {df['final_score'].max():.1f}")

print(f"\n  TOP 25 OPPORTUNITIES (no LLM, pure signal):")
for i, g in df.head(25).iterrows():
    print(f"\n  #{i+1} [Score: {g['final_score']}] ({g['post_count']} posts)")
    print(f"    {g['rep_title'][:80]}")
    print(f"    Pain:{g['pain_norm']:.2f} Eng:{g['eng_norm']:.2f} Intent:{g['intent_norm']:.2f} Build:{g['build_norm']:.2f} Valid:{g['valid_norm']:.2f}")
    print(f"    Subs: {g['top_subreddits'][:50]} | ^{g['total_upvotes']} {g['total_comments']}c")
    if isinstance(g.get("sample_titles"), str) and g["sample_titles"]:
        print(f"    Samples: {g['sample_titles'][:90]}")

print(f"\n{'='*65}")
print(f" ALL CSVs for run {RUN_ID}:")
for step_n, label in [(3,"raw"),(4,"clean"),(5,"pain"),(6,"engagement"),
                       (7,"sentiment"),(8,"buildable"),(9,"filtered"),
                       (10,"grouped"),(11,"final")]:
    fname = csv_name(step_n, label)
    if os.path.exists(fname):
        rows = len(pd.read_csv(fname))
        print(f"  {fname:45s} {rows:>6,} rows")
print(f"{'='*65}")